# SaaS Conversion Funnel Analysis — Revenue Leakage & Prioritized Recommendations
**Dataset:** User Funnels Dataset (Kaggle) · 17,175 records  
**Funnel stages:** Homepage → Product Page → Cart → Checkout → Purchase  
**Goal:** Identify where users drop off, quantify the revenue impact of each bottleneck, and deliver prioritized recommendations — the same workflow I apply for SaaS product and growth analytics clients.

---

## 1. Setup & imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f8f6',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   1.0,
    'font.family':      'DejaVu Sans',
    'axes.labelcolor':  '#444',
    'xtick.color':      '#666',
    'ytick.color':      '#666',
})

GREEN  = '#1D9E75'
BLUE   = '#378ADD'
AMBER  = '#EF9F27'
RED    = '#E24B4A'
GRAY   = '#888780'

STAGE_ORDER  = ['homepage', 'product_page', 'cart', 'checkout', 'purchase']
STAGE_LABELS = ['Homepage', 'Product Page', 'Cart', 'Checkout', 'Purchase']

print('Setup complete.')

## 2. Load & inspect

In [ ]:
df = pd.read_csv('user_data.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Nulls:\n{df.isnull().sum()}')
print(f'\nStages: {df["stage"].unique().tolist()}')
print(f'Conversion values: {df["conversion"].unique().tolist()}')
print(f'\nDistribution:')
print(df.groupby('stage')['conversion'].value_counts().to_string())
df.head()

## 3. Funnel metrics calculation

The dataset has one row per user per stage, with a boolean `conversion` flag indicating whether the user proceeded to the next step.

Key metrics I'll compute for each stage:
- **Total users** who entered the stage
- **Converted users** who passed through
- **Conversion rate** = converted / total
- **Drop-off rate** = 1 - conversion rate
- **Cumulative conversion** relative to the top of funnel (homepage)

I also compute the **step-to-step loss** — how many users were gained at one stage but lost before the next. This is different from just looking at each stage's drop-off in isolation.

In [ ]:
totals    = df.groupby('stage').size().reindex(STAGE_ORDER)
converted = df[df['conversion'] == True].groupby('stage').size().reindex(STAGE_ORDER)
dropoff   = totals - converted

conv_rate    = (converted / totals * 100).round(2)
dropoff_rate = (dropoff / totals * 100).round(2)

# Cumulative conversion relative to homepage
top = totals['homepage']
cumulative = (converted / top * 100).round(3)

funnel = pd.DataFrame({
    'users_entered':   totals,
    'users_converted': converted,
    'users_lost':      dropoff,
    'conv_rate_pct':   conv_rate,
    'dropoff_rate_pct':dropoff_rate,
    'cumulative_pct':  cumulative,
})

print('=== Funnel summary ===')
print(funnel.to_string())

# Step-to-step user losses (users gained at stage N but lost before stage N+1)
print('\n=== Step-to-step transitions ===')
for i in range(len(STAGE_ORDER) - 1):
    curr = STAGE_ORDER[i]
    nxt  = STAGE_ORDER[i + 1]
    lost = converted[curr] - converted[nxt]
    pct  = lost / converted[curr] * 100
    print(f'  {curr:15} → {nxt:15}: {lost:>5,} users lost ({pct:.1f}%)')

## 4. Funnel visualization

Two views: (1) the classic funnel showing absolute volume at each stage, and (2) conversion rates per step to make the severity of each drop immediately visible.

In [ ]:
# Color each bar by health: green = above benchmark, red = critical
# Benchmarks: homepage 45%, product_page 35%, cart 65%, checkout 75%
BENCHMARKS = {'homepage': 45, 'product_page': 35, 'cart': 65, 'checkout': 75, 'purchase': 50}
bar_colors = [GREEN if conv_rate[s] >= BENCHMARKS[s] else RED for s in STAGE_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: funnel bars (volume)
ax = axes[0]
bars = ax.barh(STAGE_LABELS[::-1], totals.values[::-1],
               color=[bar_colors[i] for i in range(len(STAGE_ORDER)-1, -1, -1)],
               height=0.55, alpha=0.85)
for bar, val in zip(bars, totals.values[::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color='#444')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_title('Users at each funnel stage', fontsize=12, pad=10)
ax.set_xlabel('Number of users')

# Right: conversion rate per step
ax2 = axes[1]
bars2 = ax2.bar(STAGE_LABELS, conv_rate.values, color=bar_colors, width=0.55, alpha=0.85)
# Benchmark line markers
for i, stage in enumerate(STAGE_ORDER):
    ax2.plot([i - 0.27, i + 0.27], [BENCHMARKS[stage], BENCHMARKS[stage]],
             color='#333', linewidth=1.5, linestyle='--')
for bar, val in zip(bars2, conv_rate.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', fontsize=9, color='#444')
ax2.set_ylim(0, 115)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax2.set_title('Conversion rate per stage (dashed = benchmark)', fontsize=12, pad=10)

green_p = mpatches.Patch(color=GREEN, alpha=0.85, label='Above benchmark')
red_p   = mpatches.Patch(color=RED,   alpha=0.85, label='Below benchmark')
ax2.legend(handles=[green_p, red_p], fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('fig_funnel_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Drop-off waterfall — where are users leaving?

I want to show not just the rate but the absolute number of users lost at each transition. This reframes the problem in business terms: every row of the chart is a lost opportunity.

In [ ]:
transitions = [
    'HP → Product', 'Product → Cart',
    'Cart → Checkout', 'Checkout → Purchase'
]

users_lost_per_step = []
for i in range(len(STAGE_ORDER) - 1):
    curr = STAGE_ORDER[i]
    nxt  = STAGE_ORDER[i + 1]
    users_lost_per_step.append(int(converted[curr] - converted[nxt]))

# Color by severity: >5000 red, >1000 amber, else gray
step_colors = [RED if v > 5000 else AMBER if v > 500 else GRAY
               for v in users_lost_per_step]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(transitions, users_lost_per_step, color=step_colors, width=0.55, alpha=0.9)

for bar, val in zip(bars, users_lost_per_step):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}', ha='center', fontsize=10, color='#333', fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_title('Users lost at each funnel transition', fontsize=13, pad=12)
ax.set_ylabel('Users lost')

plt.tight_layout()
plt.savefig('fig_dropoff_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key observation: Homepage → Product Page loses 7,485 users (largest absolute loss).')
print('However, Cart → Checkout and Checkout → Purchase have the worst RATES (92%),')  
print('making them the priority for conversion optimization.')

## 6. Benchmark comparison

Raw conversion rates don't tell the full story without context. I'm comparing against SaaS industry benchmarks from Baymard Institute and HubSpot (2023).

This is where the narrative shifts from "here's what the data shows" to "here's why it matters".

In [ ]:
bench_steps  = ['HP → Product', 'Product → Cart', 'Cart → Checkout', 'Checkout → Purchase']
our_rates    = [conv_rate['homepage'], conv_rate['product_page'],
                conv_rate['cart'], conv_rate['checkout']]
bench_rates  = [45, 35, 65, 75]

x = np.arange(len(bench_steps))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 4.5))

our_colors = [GREEN if o >= b else RED for o, b in zip(our_rates, bench_rates)]
ax.bar(x - w/2, our_rates,   width=w, color=our_colors, alpha=0.85, label='Our funnel')
ax.bar(x + w/2, bench_rates, width=w, color=GRAY,       alpha=0.5,  label='SaaS benchmark')

for i, (our, bm) in enumerate(zip(our_rates, bench_rates)):
    diff = our - bm
    label = f'+{diff:.1f}pp' if diff >= 0 else f'{diff:.1f}pp'
    color = GREEN if diff >= 0 else RED
    ax.text(i, max(our, bm) + 2.5, label, ha='center', fontsize=9,
            color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(bench_steps)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.set_ylim(0, 115)
ax.set_title('Our conversion rates vs SaaS industry benchmarks', fontsize=13, pad=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('fig_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nBenchmark gap summary:')
for step, our, bm in zip(bench_steps, our_rates, bench_rates):
    gap = our - bm
    status = 'ABOVE' if gap >= 0 else 'BELOW'
    print(f'  {step:25} {our:5.1f}% vs {bm:4.0f}%  → {status} by {abs(gap):.1f}pp')

## 7. Revenue leakage — financial impact per stage

This is the section that makes the analysis actionable for business stakeholders.

The question I'm answering: **if we improved each stage by 10 percentage points, what would the monthly revenue impact be?**

Assumptions:
- Average ticket: $49/month (standard SaaS basic plan)
- 10,000 monthly visitors (consistent with the dataset top-of-funnel)
- Only one stage changes at a time — all others remain at current rates

In [ ]:
TICKET   = 49    # USD/month
VISITORS = 10_000

current_buyers  = int(converted['purchase'])
current_revenue = current_buyers * TICKET
print(f'Baseline: {current_buyers} buyers × ${TICKET} = ${current_revenue:,}/month')

scenarios = []
for i, stage in enumerate(STAGE_ORDER[:-1]):
    curr_rate = conv_rate[stage] / 100
    improved  = min(curr_rate + 0.10, 1.0)  # +10pp, capped at 100%

    # Walk the full funnel with the improved rate at this stage
    buyers = VISITORS
    for j, s in enumerate(STAGE_ORDER[:-1]):
        rate = improved if j == i else conv_rate[s] / 100
        buyers *= rate
    buyers = round(buyers)

    uplift_mo  = (buyers - current_buyers) * TICKET
    uplift_yr  = uplift_mo * 12
    roi_factor = uplift_yr / (uplift_yr * 0.1) if uplift_yr > 0 else 0  # rough 10% dev cost

    scenarios.append({
        'stage':        STAGE_LABELS[i],
        'current_rate': round(conv_rate[stage], 1),
        'improved_rate':round(improved * 100, 1),
        'new_buyers':   buyers,
        'uplift_mo':    uplift_mo,
        'uplift_yr':    uplift_yr,
    })

df_scenarios = pd.DataFrame(scenarios)
print('\nRevenue uplift if each stage improves +10pp:')
print(df_scenarios[['stage','current_rate','improved_rate','new_buyers','uplift_mo','uplift_yr']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

uplift_vals  = df_scenarios['uplift_yr'].values
stage_names  = df_scenarios['stage'].values
# Priority coloring: highest uplift = red (urgent), lower = amber/green
uplift_colors = [RED, AMBER, AMBER, GREEN]

bars = ax.bar(stage_names, uplift_vals, color=uplift_colors, width=0.55, alpha=0.9)
for bar, val in zip(bars, uplift_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'${val:,.0f}/yr', ha='center', fontsize=9, color='#333', fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Annual revenue uplift per +10pp improvement by stage', fontsize=13, pad=12)
ax.set_ylabel('Revenue uplift (USD/year)')

plt.tight_layout()
plt.savefig('fig_revenue_leakage.png', dpi=150, bbox_inches='tight')
plt.show()

# Priority ranking
ranked = df_scenarios.sort_values('uplift_yr', ascending=False)
print('\nPriority ranking by revenue impact:')
for rank, (_, row) in enumerate(ranked.iterrows(), 1):
    print(f'  #{rank} {row["stage"]:15} → ${row["uplift_yr"]:>8,.0f}/year')

## 8. Overall conversion vs benchmark

Putting it all together: how does our end-to-end conversion compare to the market?

In [ ]:
overall_conv  = converted['purchase'] / totals['homepage'] * 100
benchmark_low = 2.0   # SaaS p25
benchmark_mid = 3.0   # SaaS median

fig, ax = plt.subplots(figsize=(7, 3.5))

categories = ['Our funnel', 'SaaS p25 (2%)', 'SaaS median (3%)']
values     = [overall_conv, benchmark_low, benchmark_mid]
colors_ov  = [RED, GRAY, GRAY]

bars = ax.bar(categories, values, color=colors_ov, width=0.45, alpha=0.85)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}%', ha='center', fontsize=10, color='#333', fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_title('End-to-end conversion rate vs SaaS benchmarks', fontsize=13, pad=12)
ax.set_ylim(0, 4)

gap = benchmark_mid - overall_conv
ax.annotate(f'{gap:.2f}pp gap to median',
            xy=(2, benchmark_mid), xytext=(1.5, 2.5),
            arrowprops=dict(arrowstyle='->', color=RED),
            fontsize=9, color=RED)

plt.tight_layout()
plt.savefig('fig_overall_conversion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Our overall conversion: {overall_conv:.3f}%')
print(f'Gap to SaaS median (3%): {gap:.2f}pp')
print(f'If we reached median: {round(VISITORS * 0.03)} buyers/month = ${round(VISITORS * 0.03 * TICKET):,}/month')

## 9. Summary & recommendations

Consolidating findings into a prioritized action plan.

In [ ]:
print('=' * 65)
print('EXECUTIVE SUMMARY — CONVERSION FUNNEL ANALYSIS')
print('=' * 65)
print(f'  Top-of-funnel visitors:    {int(totals["homepage"]):>10,}')
print(f'  Bottom-of-funnel buyers:   {int(converted["purchase"]):>10,}')
print(f'  Overall conversion rate:   {overall_conv:>9.3f}%')
print(f'  SaaS median benchmark:     {benchmark_mid:>9.1f}%')
print(f'  Gap to benchmark:          {gap:>9.2f}pp')
print()
print('  STAGE HEALTH:')
for s, label in zip(STAGE_ORDER, STAGE_LABELS):
    bm     = BENCHMARKS[s]
    rate   = conv_rate[s]
    status = 'OK  ' if rate >= bm else 'CRITICAL'
    print(f'    {label:15} {rate:5.1f}% (benchmark {bm:3.0f}%)  [{status}]')
print()
print('  PRIORITY ACTIONS (ranked by revenue impact):')
print('  #1 Checkout (92% drop-off, $151K/yr opportunity)')
print('     → Guest checkout, show total cost upfront, reduce form fields')
print('  #2 Cart (70% drop-off, $86K/yr opportunity)')
print('     → Add trust signals, returns policy, save-for-later option')
print('  #3 Product Page (50% drop-off, $76K/yr opportunity)')
print('     → Improve CTA clarity, add social proof, optimize pricing presentation')
print('=' * 65)
print(f'\n  If all three stages reach benchmark: ~{round(VISITORS*0.03)} buyers/mo = ${round(VISITORS*0.03*TICKET):,}/mo')

---
*Analysis by [Your Name] | Tools: Python · pandas · matplotlib | Dataset: User Funnels (Kaggle) | Benchmarks: Baymard Institute, HubSpot SaaS 2023*